# Non-Negative Least Square

Non-Negative Least Square or NNLS is variant of Linear Regression where we force every coefficient to be positive. And fo rthis purpose we are goin to use Coordinate Descent.

### Coordinate Descent

Coordinate descent is an optimization technique where we have to minimize along a coordinate and then switch to another coordinate to minmize and keep minimizing until we reach the minimum point.

<center> <img src = 'images/coordinate-descent.png'></center>

First we diefine our loss function for this we use RSS,
$$\mathcal{L}_l = RSS = \sum_{i=1}^{n} {(y_i - \hat{y}_i)}^2 $$

And at this point instead of differentiating it, with total $b$ we differentiate only one coordiante $b_j$, $\frac{d \mathcal{L}_l}{d b_j}$

$$\frac{d \mathcal{L}_l}{d b_j} = \frac{d }{d b_j} ((y_i - \hat{y}_i)^2$$

For $\frac{d }{d b_j}(y_i - \hat{y}_i)^2$ we get,
$$ \displaylines{ \frac{d }{d b_j} \sum_{i=1}^{n} {(y_i - \hat{y}_i)}^2 \\
= \frac{d }{d b_j} \sum_{i=1}^{n} (y_i - \sum_{k=0}^m b_kx_i)^2 \\
=-2 \sum_{i=1}^nx_i^{(j)}(y_i - \sum_{k=0}^m b_kx_i)^2}$$

Here, $x_i^{(j)}$ is the $j^th$ column associated with $b_j$ coefficient.

$$\displaylines{
= -2 \sum_{i=1}^nx_i^{(j)}(y_i - \sum_{k=0,k \neq j}^m b_kx_i^{(k)} - b_jx_i^{(j)})\\
= -2 \sum_{i=1}^nx_i^{(j)}(y_i - \sum_{k=0,k \neq j}^m b_kx_i^{(k)}) -2\sum_{i=1}^nb_j(x_i^{(j)})^2}$$

lets consider  $\sum_{i=1}^nx_i^{(j)}(y_i - \sum_{k=0,k \neq j}^m b_kx_i^{(k)})=\rho_j$ and $2\sum_{i=1}^n b_j (x_i^{(j)})^2 = 2b_j\sum_{i=1}^n(x_i^{(j)})^2 = 2b_j z_j$

So, $$\frac{d }{d b_j}RSS = -2\rho_j + 2b_j z$$

In order to minimize we set $\frac{d }{d b_j}RSS = 0$ and solve for $b_j$

$$\frac{d }{d b_j}RSS = -2\rho_j + 2b_j z_j = 0$$
$$ b_j = \frac{\rho_j}{z_j}$$

Now we set a condition every time we update our coefficient. we set condition to be $0$ whenever the coefficient heads towards negative value.

$$ \hat{b_j} = \begin{cases}
\frac{\rho_j}{ z_j} & \text{if} \rho_j > 0 \\
0 & \text{if} \rho_j \le 0 \\
\end{cases} $$

we are going to update each coefficient with this equation until the maximum change of any coefficient through a loop is less than a threshold vlaue.

If we want to rearange these equation in matrix form we will get,
$$ \rho_j = X_j^T(Y - X_kb_k) = X_j^TY-X_j^TX_kb_k $$
$$ z_j = X_j^TX_j $$

Lets implement this in a function `nnls`

In [28]:
from modules.mylinalg import matrix_rank, transpose, cholesky_inverse, matmul, svdvals, to2dim, flatten, dot, shape
from modules.myprocessing import  validate

In [30]:
def nnls(X,y,L2_penalty = 0,gram=None):

    X,y = validate(X,y)
    
    if shape(y)[1] != 1:
        raise Exception ('please provide only one target variable')
    
    max_iter = 10000
    tol =0.0001
    warm_start = False
    random.seed(10)
    b = None
    selection = 'cyclic'
    b = [ 0 for i in range(len(X[0]))]

    gram = matmul(transpose(X),X) if not gram else gram

    loop_max_step = 100
    
    while loop_max_step > tol:
        max_step = 0
       
        idx_set = set(range(len(b)))
        
        j = 0
        iter = 0
        if iter > max_iter:
            break
        
        while idx_set:
            iter += 1
            if selection == 'random':
                j = random.choice(idx_set)
            idx_set.discard(j)
            
            b_j = b[j]
            X_j = [x[j] for x in X]
            XT_jY = dot(X_j,y)
            
            b_k=[]
            XT_jX_k = []
            for k in range(len(b)):
                if k != j:
                    b_k.append(b[k])
                    XT_jX_k.append(gram[j][k])

            
            XT_jX_kb_k = dot(XT_jX_k,b_k)
            rho_j = XT_jY - XT_jX_kb_k
           
            z_j = gram[j][j]

            if j == 0 or rho_j > 0:
                b_j_hat = rho_j/(L2_penalty+z_j)
            else:
                b_j_hat = 0

            step = abs(b_j_hat - b_j)
            max_step = max(step, max_step)
            b[j] = b_j_hat

            if selection == 'cyclic':
                j += 1
        loop_max_step = max_step
    return b

In [32]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=50, n_features=5, noise=8, tail_strength=0.5, random_state=10)
X[:,2] = -X[:,2]
y = y.reshape(-1,1)

In [34]:
nnls(X,y)

[28.912946964969425,
 25.176325590437976,
 0,
 65.14645620099994,
 11.179896930067212]

We cna see our $3^{rd}$ coefficient is $0$ which had a negative correlation with the target variable